In [1]:
from PIL import Image
import cv2
import torch
import function.utils_rotate as utils_rotate
import function.helper as helper



In [2]:
 #Load YOLO models
yolo_LP_detect = torch.hub.load('yolov5', 'custom', path='model/LP_detector.pt', force_reload=True, source='local')
yolo_license_plate = torch.hub.load('yolov5', 'custom', path='model/LP_ocr.pt', force_reload=True, source='local')
yolo_license_plate.conf = 0.60



YOLOv5  v6.1-179-gf3fecf94 torch 2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)

e:\venv\prj\licensePlateTese\License-Plate-Recognition\yolov5\models\experimental.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on G

In [3]:
# Load image
img_file = "test_image/image.png"
img = cv2.imread(img_file)

# Detect license plates
plates = yolo_LP_detect(img, size=640)
list_plates = plates.pandas().xyxy[0].values.tolist()

# Default to full image if no plate detected
if not list_plates:
    plate_img = img
else:
    # Assume only one plate
    x1, y1, x2, y2 = map(int, list_plates[0][:4])
    plate_img = img[y1:y2, x1:x2]

# Try deskew and OCR
plate_number = "unknown"
for cc in range(2):
    for ct in range(2):
        rotated = utils_rotate.deskew(plate_img, cc, ct)
        plate_number = helper.read_plate(yolo_license_plate, rotated)
        if plate_number != "unknown":
            break
    if plate_number != "unknown":
        break

print(plate_number)


e:\venv\prj\licensePlateTese\License-Plate-Recognition\yolov5\models\common.py:565: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
e:\venv\prj\licensePlateTese\License-Plate-Recognition\yolov5\models\common.py:565: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


29T8-2843
